### Step 1 : Include Libraries and Initialize Spark Session

In [7]:
spark.stop()

In [3]:
from pyspark.sql.functions import regexp_extract

In [1]:
#Import necessary libraries and initialize Spark Session

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("AccessLogStreaming")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/19 20:37:55 WARN Utils: Your hostname, Stefans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.118 instead (on interface en0)
26/09/19 20:37:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/stefangarevski/.ivy2.5.2/cache
The jars for the packages stored in: /Users/stefangarevski/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4faa105b-378a-4f4d-91b4-69cca82117b5;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafk

Spark version: 4.2.0


## Use-Case : Tracking Server Access Log <a class="anchor" name="use-case"></a>
For this case, a server is going to continuously send a records of a host who is trying to access some endpoint (url) from the web server. This data will be send from a kafka producer (<code>7-Example-Kafka-Producer-AccessLog.ipynb</code>) which is reading the data from a txt file in the dataset provided (<code>access_log.txt</code>).

Each line contains some valuable information such as:

1. Host
2. Timestamp
3. HTTP method
4. URL endpoint
5. Status code
6. Protocol
7. Content Size

The goal here is to perform some real time queries from this stream of data and be able to output the results in multiple ways.

### Step 2 : Load Kafka Stream 
Use the <code>readStream</code> to load data from the Kafka Producer <strong>7-Example-Kafka-Producer-AccessLog.ipynb</strong>

<a class="anchor" id="lab-task-1"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">1. Lab Task: </strong> 
    Write the code below to readStream from the the producer into <code>df_urls</code> dataframe.
</div>

In [2]:
topic = "access_log"

df_urls = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", topic)
    .option("startingOffsets", "latest")
    .load()
)

print(df_urls.isStreaming)

True


## Data Preparation <a class="anchor" name="data-prep"></a>
We need to convert the data from the message in order to perform some queries. The steps to parse the data are:

1. Get message as a string from <code>value</code> which is binary.
2. Implement some regular expressions to capture specific fields in the message which is a line from the access log.
3. Extract the values using the regular expressions to create the dataframe.

<a class="anchor" id="lab-task-2"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">2. Lab Task: </strong> 
    Complete the code below to use regular expression to wrangle the data.
</div>

In [4]:
# Get value of the kafka message
log_lines = df_urls.selectExpr("CAST(value AS STRING)")

# Parse out the common log format to a DataFrame
statusExp = r'\s(\d{3})\s'
generalExp = r'\"(\S+)\s(\S+)\s*(\S*)\"'
hostExp = r'(^\S+\.[\S+\.]+\S+)\s'

df_logs = log_lines.select(
    regexp_extract('value', hostExp, 1).alias('host'),
    regexp_extract('value', generalExp, 1).alias('method'),
    regexp_extract('value', generalExp, 2).alias('endpoint'),
    regexp_extract('value', generalExp, 3).alias('protocol'),
    regexp_extract('value', statusExp, 1).cast('integer').alias('status')
)

df_logs.printSchema()

root
 |-- host: string (nullable = true)
 |-- method: string (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status: integer (nullable = true)



## Data Streaming Processing 

<a class="anchor" id="lab-task-2"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">3. Lab Task: </strong> 
    Write a DataFrame query to filter out those requests that were not successful using <code>status !=200</code> filter.
</div>

In [5]:
# 1. DF that filters those requests that were not successful (status != 200)
unsucess_df = df_logs.filter(df_logs.status != 200)

<a class="anchor" id="lab-task-3"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">4. Lab Task: </strong> 
    Write a DataFrame query count the number of requests by access status code
</div>

In [6]:
# 2. DF that keeps a running count of every access by status code
status_count_df = df_logs.groupBy("status").count()

## Output sink <a class="anchor" name="output-sink"></a>
Before starting this section, run the kafka producer that will send the data from the access log file.

In [7]:
# Create function to show values received from input dataframe
def foreach_batch_function(df, epoch_id):
    df.show(20,False)

#### Display stream output in notebook <a class="anchor" name="foreachBatch"></a>

In [15]:
# Write output of status_count_df in output cell using the foreach_batch_function
# Control the amount of times output is displayed with trigger function
query1 = status_count_df.writeStream.outputMode("complete")\
        .foreachBatch(foreach_batch_function)\
        .start()

26/09/19 20:59:29 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/9l/03xt3yvj2gg2s7lfps5hf6gh0000gn/T/temporary-b9922627-5d3c-40fe-b946-7e2b8684eac5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/09/19 20:59:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/09/19 20:59:29 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
                                                                                

+------+-----+
|status|count|
+------+-----+
+------+-----+



+------+-----+
|status|count|
+------+-----+
|200   |4    |
+------+-----+



+------+-----+
|status|count|
+------+-----+
|200   |9    |
+------+-----+



+------+-----+
|status|count|
+------+-----+
|301   |1    |
|200   |12   |
+------+-----+



...

+------+-----+
|status|count|
+------+-----+
|301   |4    |
|404   |1    |
|200   |101  |
+------+-----+



+------+-----+
|status|count|
+------+-----+
|301   |4    |
|404   |1    |
|200   |105  |
+------+-----+



+------+-----+
|status|count|
+------+-----+
|301   |4    |
|404   |1    |
|200   |109  |
+------+-----+



+------+-----+
|status|count|
+------+-----+
|301   |4    |
|404   |1    |
|200   |113  |
+------+-----+



+------+-----+
|status|count|
+------+-----+
|301   |4    |
|404   |1    |
|200   |117  |
+------+-----+



In [16]:
query1.stop()

26/09/19 21:00:34 WARN Shell: Interrupted while joining on: Thread[Thread-80990,5,]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1315)
	at java.base/java.lang.Thread.join(Thread.java:1383)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1106)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1066)
	at org.apache.hadoop.util.Shell.run(Shell.java:960)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1285)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1380)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1362)
	at org.apache.hadoop.fs.FileUtil.readLink(FileUtil.java:225)
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileLinkStatusInternal(RawLocalFileSystem.java:1314)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1303)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatus(RawLocalFileSy

<a class="anchor" id="lab-task-4"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">5. Lab Task: </strong> 
    Write the stream output to the <strong>memory sink</strong> and display the result using <strong>spark SQL</strong>
</div>

In [17]:
query2 = (
    status_count_df.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("status_counts")
    .start()
)

26/09/19 21:01:15 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/9l/03xt3yvj2gg2s7lfps5hf6gh0000gn/T/temporary-3cabb19f-f45f-47e6-932a-81e48c5d9fcf. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/09/19 21:01:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/09/19 21:01:15 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
                                                                                

In [18]:
spark.sql("SELECT * FROM status_counts").show()

[Stage 416:=========>  (157 + 10) / 200][Stage 417:>                (0 + 0) / 1]

+------+-----+
|status|count|
+------+-----+
|   200|   35|
+------+-----+

